# Step 0: Import requisite libraries

In [184]:
import pandas as pd
import numpy as np
from sklearn import preprocessing, ensemble, tree
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import matplotlib as plt
import seaborn

# Step 1: Import and cleaning of data

In [185]:
df = pd.read_csv("Baza_customer_Telecom_v2.csv")
df["CHURN"].value_counts()

CHURN
No     7904
Yes     549
Name: count, dtype: int64

In [186]:
print(f"Length of dataset before cleaning: {len(df)}")
# Cleaning for PID
df = df.loc[df["PID"].notnull() & df["PID"].str.fullmatch(r"[0-9]*(SMB)?")] # Some PIDs were scientific notation (invalid); valid format is integer optionally followed by "SMB"

# Cleaning for value segment
df = df.loc[df["CRM_PID_Value_Segment"].notnull()]
df["CRM_PID_Value_Segment"] = df["CRM_PID_Value_Segment"].replace("Sliver", "Silver") # Fix assumed typo in one entry

# Cleaning for effective segment
df = df.loc[df["EffectiveSegment"].notnull()]

# Cleaning for ZIP
df = df.loc[df["Billing_ZIP"] % 1 == 0] # Remove non-integer ZIP codes (invalid; includes nulls)

# Cleaning for KA_name
df = df.loc[df["KA_name"].notnull()]
df["KA_name"] = df["KA_name"].replace("AD?", "AD") # Grouped AD? with AD under assumption that unsure attribution is correct

# Cleaning for active_subscribers
df = df.loc[df["Active_subscribers"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for not_active_subscribers
df["Not_Active_subscribers"] = df["Not_Active_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none inactive; minimum non-NaN value is 1)

# Cleaning for suspended_subscribers
df["Suspended_subscribers"] = df["Suspended_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none suspended; minimum non-NaN value is 1)

# Cleaning for total_SUBs
df = df.loc[df["Total_SUBs"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for AvgMobileRevenue
df = df.loc[df["AvgMobileRevenue "].notnull()] # Redundant (no nulls) but included for consistency
df = df.rename(columns={"AvgMobileRevenue ": "AvgMobileRevenue"}) # Remove trailing space from column name

# Cleaning for AvgFIXRevenue
df = df.loc[df["AvgFIXRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for TotalRevenue
df = df.loc[df["TotalRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for ARPU
df = df.loc[df["ARPU"].notnull()]

# Cleaning for CHURN
df = df.loc[df["CHURN"].isin(["No", "Yes"])] # Remove invalid CHURN values (only valid values are "No" and "Yes"); redundant (no invalids) but included for consistency

print(f"Length of dataset after cleaning: {len(df)}") # Small length change, hence valid to discard nulls/invalid entries

# Source - https://stackoverflow.com/a/48035423
# Posted by BENY, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-02, License - CC BY-SA 4.0

df.apply(lambda x : pd.factorize(x)[0]).corr(method='pearson', min_periods=1)

Length of dataset before cleaning: 8453
Length of dataset after cleaning: 8419


,PID,CRM_PID_Value_Segment,EffectiveSegment,Billing_ZIP,KA_name,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,CHURN
PID,1.000000,0.192726,0.256025,0.167651,0.529756,0.121228,0.084432,0.043259,0.147022,0.242257,0.093210,0.244694,0.350940,0.012618
CRM_PID_Value_Segment,0.192726,1.000000,0.631015,0.030139,-0.041252,0.498979,0.237404,0.105070,0.482523,0.676847,0.063799,0.676468,0.300284,0.042259
EffectiveSegment,0.256025,0.631015,1.000000,0.036593,-0.017018,0.425958,0.281537,0.095870,0.452980,0.539749,0.072822,0.541325,0.233199,0.038939
Billing_ZIP,0.167651,0.030139,0.036593,1.000000,0.126296,0.019873,0.014832,-0.000522,0.021090,0.016185,0.057988,0.011743,0.031975,0.018897
KA_name,0.529756,-0.041252,-0.017018,0.126296,1.000000,-0.041324,0.009309,0.006503,-0.012572,-0.015707,0.031672,-0.012891,0.183404,0.020531
Active_subscribers,0.121228,0.498979,0.425958,0.019873,-0.041324,1.000000,0.155878,0.070501,0.732803,0.662186,0.022676,0.660838,0.108241,0.075000
Not_Active_subscribers,0.084432,0.237404,0.281537,0.014832,0.009309,0.155878,1.000000,0.075496,0.545210,0.194267,0.070973,0.198449,0.099186,0.016040
Suspended_subscribers,0.043259,0.105070,0.095870,-0.000522,0.006503,0.070501,0.075496,1.000000,0.126213,0.110787,-0.001968,0.110039,0.067284,0.014210
Total_SUBs,0.147022,0.482523,0.452980,0.021090,-0.012572,0.732803,0.545210,0.126213,1.000000,0.567091,0.055602,0.570308,0.144431,0.048840
AvgMobileRevenue,0.242257,0.676847,0.539749,0.016185,-0.015707,0.662186,0.194267,0.110787,0.567091,1.000000,0.097648,0.988084,0.398952,0.063913


# Step 2: Data Preprocessing

In [187]:
X = df.drop(columns=["CHURN"])
y = df["CHURN"].map({"No": 0, "Yes": 1}) # Convert CHURN to binary values (0 for No, 1 for Yes)

X["PID_is_SMB"] = X["PID"].str.endswith("SMB").astype("string") # Create binary feature indicating if PID ends with "SMB"
X["PID"]=X["PID"].str.replace("SMB", "").astype(int) # Remove "SMB" from PID and convert to integer

# Definition of categorical and numerical features for preprocessing
cat_features = ["CRM_PID_Value_Segment", "EffectiveSegment", "KA_name", "PID_is_SMB"]
num_features = ["Billing_ZIP", "Active_subscribers", "Not_Active_subscribers", "Suspended_subscribers", "Total_SUBs", "AvgMobileRevenue", "AvgFIXRevenue", "TotalRevenue", "ARPU"]

# Selection of features for model training

# train_features = cat_features + num_features

train_features = [
    "PID",
    "PID_is_SMB",
    "CRM_PID_Value_Segment", 
    "Active_subscribers",
    "AvgMobileRevenue",
    "TotalRevenue"
    ]

X = X[train_features]

# Definition of ColumnTransformer for preprocessing
c_transformer = ColumnTransformer(
    transformers=[
        ("categorical", preprocessing.OneHotEncoder(handle_unknown="ignore"), list(set(cat_features) & set(train_features)) ),
        ("numerical", "passthrough", list(set(num_features) & set(train_features)) )
    ]
)

print(f"Column transformer has categorical features {list(set(cat_features) & set(train_features))} and numerical features {list(set(num_features) & set(train_features))}")

# Model pipeline
clf_p = Pipeline([
    ("preprocessor", c_transformer),
    ("classifier", ensemble.RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit model
clf_p.fit(X_train, y_train)

# Predict on test set
y_pred_p = clf_p.predict_proba(X_test)
t_df = pd.DataFrame(X_test)
t_df["Predicted_Churn_Probability"] = y_pred_p[:, 1] # Get probabilities for positive class (CHURN = Yes)

# Evaluate model performance
y_pred = clf_p.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
t_df.sort_values(by="Predicted_Churn_Probability", ascending=False).head(20)

Column transformer has categorical features ['CRM_PID_Value_Segment', 'PID_is_SMB'] and numerical features ['TotalRevenue', 'Active_subscribers', 'AvgMobileRevenue']
              precision    recall  f1-score   support

           0       0.94      0.92      0.93      1575
           1       0.12      0.17      0.14       109

    accuracy                           0.87      1684
   macro avg       0.53      0.54      0.54      1684
weighted avg       0.89      0.87      0.88      1684

[[1447  128]
 [  91   18]]


,PID,PID_is_SMB,CRM_PID_Value_Segment,Active_subscribers,AvgMobileRevenue,TotalRevenue,Predicted_Churn_Probability
1423,40370833,False,Bronze,7,87.83,87.83,1.000
695,123006700,False,Bronze,5,64.50,64.50,1.000
693,123749287,False,Bronze,5,64.50,64.50,1.000
2270,108691439,False,Bronze,8,118.33,118.33,1.000
5644,115799835,False,Bronze,7,74.33,74.33,1.000
7520,123729833,False,SME,7,188.17,188.17,1.000
6202,201765222,False,Silver,6,103.67,103.67,1.000
7175,115813676,False,Bronze,2,63.17,63.17,1.000
1388,112651920,False,Bronze,3,86.67,86.67,1.000
5769,160086318,False,Bronze,2,79.67,79.67,1.000


In [ ]:
# Testing per-group
test_segment = "Bronze"
segment_test = y_test[X_test["CRM_PID_Value_Segment"] == test_segment]
segment_pred = y_pred[X_test["CRM_PID_Value_Segment"] == test_segment]

print(classification_report(segment_test, segment_pred))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94       286
           1       0.17      0.21      0.19        19

    accuracy                           0.89       305
   macro avg       0.56      0.57      0.56       305
weighted avg       0.90      0.89      0.89       305

